<a href="https://colab.research.google.com/github/Val-Faria/tech-challenge-ia-saude/blob/main/02_otimizacao_hiperparametros_ag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Otimização de Modelos de Diagnóstico e Interpretação com IA Generativa
## Tech Challenge - Fase 2 | IA para Devs

Este projeto dá continuidade à solução desenvolvida na fase anterior para classificação de registros clínicos relacionados à tireoide. A proposta combina Algoritmos Genéticos para otimização automática de hiperparâmetros dos modelos de Machine Learning e Large Language Models (LLMs) para interpretação dos resultados em linguagem natural.

O objetivo é melhorar o desempenho dos modelos diagnósticos por meio de técnicas evolucionárias e transformar predições, probabilidades e métricas em explicações mais acessíveis e úteis para profissionais da saúde.

A solução possui caráter acadêmico e experimental. Os resultados gerados devem ser utilizados apenas como apoio à análise clínica, não substituindo a avaliação e o julgamento de profissionais da saúde.

## 1. Preparação do Ambiente

A execução do projeto foi planejada para o Google Colab, permitindo o carregamento automático dos dados e a reprodução dos experimentos em ambiente padronizado. A célula a seguir realiza as configurações iniciais do ambiente e pode incluir a instalação de dependências necessárias para a execução das etapas de otimização por Algoritmos Genéticos e interpretação dos resultados com LLMs.


In [8]:
# Opcional - execute apenas se necessário
# !pip install openai
# !pip install transformers
# !pip install accelerate

## 2. Importação das Bibliotecas

O projeto utiliza bibliotecas para análise de dados, visualização, treinamento e avaliação de modelos de Machine Learning, além de recursos para implementação do Algoritmo Genético empregado na otimização de hiperparâmetros. O parâmetro `RANDOM_STATE` foi adotado para aumentar a reprodutibilidade dos resultados.


In [9]:
import os
import random
import sys
import urllib.request
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_COLUMN = "target"
POSITIVE_LABEL = "hypothyroid"
NEGATIVE_LABEL = "negative"

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Seed global definida: {RANDOM_STATE}")


Seed global definida: 42


## 3. Fonte de Dados e Diretórios do Projeto

O dataset hypothyroid_final.csv é carregado automaticamente do GitHub no Google Colab ou da pasta dataset/ em ambiente local. Os diretórios necessários para armazenamento dos resultados são criados durante a execução.

In [10]:
GITHUB_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "mvaraujo1977/TechCahllenge_Tireoide/"
    "Nirton_Afonso/dataset/hypothyroid_final.csv"
)

RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    PROJECT_ROOT = Path("/content/TechCahllenge_Tireoide")
    DATA_DIR = PROJECT_ROOT / "dataset"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    DATA_PATH = DATA_DIR / "hypothyroid_final.csv"

    print("Ambiente Google Colab detectado.")
    print(f"Baixando dataset do GitHub para: {DATA_PATH}")
    urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

    DATA_PATH = PROJECT_ROOT / "dataset" / "hypothyroid_final.csv"
    if not DATA_PATH.exists():
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("Dataset local não encontrado. Baixando do GitHub para a pasta dataset/ do projeto.")
        urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fonte GitHub: {GITHUB_DATA_URL}")
print(f"Dataset em uso: {DATA_PATH.resolve()}")
print(f"Diretório de modelos: {MODELS_DIR.resolve()}")
print(f"Diretório de figuras: {FIGURES_DIR.resolve()}")
print(f"Diretório de resultados: {RESULTS_DIR.resolve()}")

Ambiente Google Colab detectado.
Baixando dataset do GitHub para: /content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv
Fonte GitHub: https://raw.githubusercontent.com/mvaraujo1977/TechCahllenge_Tireoide/Nirton_Afonso/dataset/hypothyroid_final.csv
Dataset em uso: /content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv
Diretório de modelos: /content/TechCahllenge_Tireoide/models
Diretório de figuras: /content/TechCahllenge_Tireoide/reports/figures
Diretório de resultados: /content/TechCahllenge_Tireoide/reports/results


## 4. Carregamento do Dataset

O conjunto de dados será carregado para o treinamento do modelo Random Forest e para a avaliação das diferentes configurações de hiperparâmetros exploradas pelo Algoritmo Genético.

In [11]:
dados = pd.read_csv(DATA_PATH, na_values=["?", "", "NA", "NaN"])

print(f"Dataset carregado com sucesso!")
print(f"Dimensões: {dados.shape[0]} linhas × {dados.shape[1]} colunas")

display(dados.head())

Dataset carregado com sucesso!
Dimensões: 3163 linhas × 26 colunas


,target,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,thyroid_surgery,query_hypothyroid,query_hyperthyroid,pregnant,...,T3_measured,T3,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG
0,hypothyroid,72.0,M,f,f,f,f,f,f,f,...,y,0.6,y,15.0,y,1.48,y,10.0,n,NaN
1,hypothyroid,15.0,F,t,f,f,f,f,f,f,...,y,1.7,y,19.0,y,1.13,y,17.0,n,NaN
2,hypothyroid,24.0,M,f,f,f,f,f,f,f,...,y,0.2,y,4.0,y,1.00,y,0.0,n,NaN
3,hypothyroid,24.0,F,f,f,f,f,f,f,f,...,y,0.4,y,6.0,y,1.04,y,6.0,n,NaN
4,hypothyroid,77.0,M,f,f,f,f,f,f,f,...,y,1.2,y,57.0,y,1.28,y,44.0,n,NaN


## 5. Limpeza dos Dados

A limpeza adotou uma postura conservadora, adequada ao contexto médico. Registros duplicados foram removidos para reduzir viés amostral, enquanto valores extremos foram mantidos por sua possível relevância clínica. O tratamento de ausentes foi reservado ao pipeline de modelagem.

In [12]:
dados_limpos = dados.copy()
antes = len(dados_limpos)
dados_limpos = dados_limpos.drop_duplicates().reset_index(drop=True)
depois = len(dados_limpos)

print(f"Registros antes: {antes}")
print(f"Registros após remoção de duplicados: {depois}")
print(f"Duplicados removidos: {antes - depois}")

dados_limpos = dados_limpos[dados_limpos[TARGET_COLUMN].isin([NEGATIVE_LABEL, POSITIVE_LABEL])].reset_index(drop=True)

print("Distribuição após limpeza:")
display(dados_limpos[TARGET_COLUMN].value_counts().to_frame("quantidade"))

Registros antes: 3163
Registros após remoção de duplicados: 3086
Duplicados removidos: 77
Distribuição após limpeza:


,quantidade
target,
negative,2945
hypothyroid,141


In [13]:
checks = []
if "age" in dados_limpos.columns:
    checks.append({
        "variavel": "age",
        "menor_que_zero": int((dados_limpos["age"] < 0).sum()),
        "maior_que_120": int((dados_limpos["age"] > 120).sum()),
    })

for col in ["TSH", "T3", "TT4", "T4U", "FTI", "TBG"]:
    if col in dados_limpos.columns:
        checks.append({
            "variavel": col,
            "menor_que_zero": int((dados_limpos[col] < 0).sum()),
            "maior_que_120": np.nan,
        })

display(pd.DataFrame(checks))

,variavel,menor_que_zero,maior_que_120
0,age,0,0.0
1,TSH,0,NaN
2,T3,0,NaN
3,TT4,0,NaN
4,T4U,0,NaN
5,FTI,0,NaN
6,TBG,0,NaN


## 6. Separação entre Variáveis Preditoras e Variável-Alvo

Nesta etapa, o dataset é dividido em variáveis preditoras (`X`) e variável-alvo (`y`). Essa separação é essencial para estruturar o problema de aprendizado supervisionado, permitindo o treinamento do modelo Random Forest e a avaliação das diferentes combinações de hiperparâmetros exploradas pelo Algoritmo Genético.

In [14]:
X = dados_limpos.drop(columns=[TARGET_COLUMN])
y = dados_limpos[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})

print(f"Formato de X: {X.shape}")
print(f"Formato de y: {y.shape}")
print("Mapeamento: 0 = negative, 1 = hypothyroid")
display(y.value_counts().rename(index={0: NEGATIVE_LABEL, 1: POSITIVE_LABEL}).to_frame("quantidade"))

Formato de X: (3086, 25)
Formato de y: (3086,)
Mapeamento: 0 = negative, 1 = hypothyroid


,quantidade
target,
negative,2945
hypothyroid,141


## 7. Divisão dos Dados

O dataset foi particionado de forma **estratificada** para preservar a proporção original das classes:

* **Treino (70%):** Ajuste e aprendizado do modelo.
* **Validação (15%):** Otimização de hiperparâmetros (Algoritmo Genético) e seleção do modelo.
* **Teste (15%):** Avaliação final e métricas de generalização (dados inéditos).

In [15]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

splits_summary = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "quantidade": [len(X_train), len(X_val), len(X_test)],
    "percentual": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
    "positivos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
})
splits_summary["percentual"] = (splits_summary["percentual"] * 100).round(2)
display(splits_summary)

,conjunto,quantidade,percentual,positivos
0,treino,2160,69.99,99
1,validação,463,15.00,21
2,teste,463,15.00,21


## 8. Pipeline de Pré-processamento

O pré-processamento foi encapsulado em pipelines do `scikit-learn` para reduzir risco de data leakage. Assim, imputação, padronização e codificação categórica são ajustadas somente no conjunto de treino e aplicadas de forma consistente aos demais subconjuntos.

In [16]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variáveis numéricas:", numeric_features)
print("Variáveis categóricas:", categorical_features)

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(numeric_steps)
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

Variáveis numéricas: ['age', 'TSH', 'T3', 'TT4', 'T4U', 'FTI', 'TBG']
Variáveis categóricas: ['sex', 'on_thyroxine', 'query_on_thyroxine', 'on_antithyroid_medication', 'thyroid_surgery', 'query_hypothyroid', 'query_hyperthyroid', 'pregnant', 'sick', 'tumor', 'lithium', 'goitre', 'TSH_measured', 'T3_measured', 'TT4_measured', 'T4U_measured', 'FTI_measured', 'TBG_measured']


## 9. Modelagem

Como o modelo **Random Forest** apresentou o melhor desempenho preliminar, o foco desta etapa foi direcionado exclusivamente para a otimização de seus hiperparâmetros. Utilizando uma abordagem evolutiva baseada em **Algoritmo Genético**, o espaço de busca foi explorado para maximizar a capacidade de generalização do classificador, mitigando o risco de *overfitting* mapeado nas análises anteriores.

In [17]:
modelos = {
    "Random Forest": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ])
}

print("Dicionário de modelos atualizado. Foco exclusivo em:", list(modelos.keys()))

Dicionário de modelos atualizado. Foco exclusivo em: ['Random Forest']


In [18]:
def get_positive_scores(model, X_data):
    return model.predict_proba(X_data)[:, 1]

def evaluate_model(model, X_data, y_true):
    y_pred = model.predict(X_data)

    # Probabilidade da classe positiva (0.0 a 1.0) para o cálculo do AUC
    y_score = get_positive_scores(model, X_data)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_score)
    }

    return metrics, y_pred, y_score